# soamp — featurization grid: 4 experiments, 5-fold CV, on Colab GPU

CV counterpart to `02_run_experiments.ipynb`. Same 2×2 grid of peptide
representation (`rdkit_descriptors` | `peptideclm_embedding`) × organism
representation (`vocab_embedding` | `kmer_composition`), with
`attention_fusion_classifier` held fixed — but each cell is run through
`pipeline/train_cv.py`'s 5-fold cross-validation over
`data/train_folds_leiden.csv`'s Leiden-community folds, instead of
`pipeline/train.py`'s single held-out train/val/test split.

This gives a cross-fold generalization estimate over the **train split
only** — it never touches `test`, and produces no checkpoints (`.pth`
files), only a per-fold results CSV and cross-fold mean/std summary
metrics. Run `02_run_experiments.ipynb` for the real held-out test
benchmark; run this notebook for a faster, higher-variance diagnostic
(`epochs=5` per fold, not 30) of the same four representation cells.

Each cell is a committed config under `config/train/`, executed through
`pipeline/train_cv.py` — the same entrypoint a local CV run uses. This
notebook adds no training/CV loop of its own; it bootstraps the
environment, builds the one artifact that needs a GPU, and collects
results.

| run (`exp_id`) | peptide | organism |
|---|---|---|
| `rdkit_vocab_cv` | RDKit descriptors (13d) | learned embedding |
| `rdkit_kmer_cv` | RDKit descriptors (13d) | genome k-mer (340d) |
| `peptideclm_vocab_cv` | PeptideCLM (768d) | learned embedding |
| `peptideclm_kmer_cv` | PeptideCLM (768d) | genome k-mer (340d) |

All four log to wandb project `soamp`, group `featurization_grid_cv_v1`
(distinct from `02`'s `featurization_grid_v1`).

## Before you run

Run `01_smoke_overfit.ipynb` first — it checks every cell constructs and can
drive its loss to zero. A **GPU runtime** matters here only for the PeptideCLM
featurization step; the classifier itself is small. If you already ran
`02_run_experiments.ipynb` in this Drive account, its cached PeptideCLM
artifact is reused automatically below — no rebuild needed.

Set these under **Colab Secrets** (the key icon in the left sidebar), with
notebook access enabled for each:

| Secret | Needed for |
|---|---|
| `GITHUB_TOKEN` | cloning the private repo (a fine-grained, read-only PAT is enough) |
| `WANDB_API_KEY` | logging runs to Weights & Biases |
| `WANDB_ENTITY` | optional; your wandb team/username if it isn't your default |

## 1. Environment

In [ ]:
import subprocess
import sys

print("Python:", sys.version.split()[0])
try:
    gpu = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
        capture_output=True, text=True, check=True,
    ).stdout.strip()
    print("GPU:", gpu)
except (FileNotFoundError, subprocess.CalledProcessError):
    print("GPU: none detected -- set Runtime > Change runtime type > T4 GPU, then rerun.")

In [ ]:
import os

from google.colab import userdata

# Colab Secrets (key icon in the left sidebar), not a committed .env: the repo
# gitignores .env, so nothing sensitive travels with the clone.
# WANDB_API_KEY is required here -- the four runs log to wandb.
REQUIRED = {"GITHUB_TOKEN": True, "WANDB_API_KEY": True, "WANDB_ENTITY": False}

for name, required in REQUIRED.items():
    try:
        os.environ[name] = userdata.get(name)
        print(f"{name}: set")
    except Exception as e:
        if required:
            raise RuntimeError(
                f"Colab secret {name!r} is missing. Add it under the key icon "
                f"in the left sidebar and enable notebook access."
            ) from e
        print(f"{name}: not set (optional)")

In [ ]:
import os
import subprocess

REPO = "LukaJinc/soamp"
BRANCH = "main"
WORKDIR = "/content/soamp"

if not os.path.isdir(WORKDIR):
    token = os.environ["GITHUB_TOKEN"]
    result = subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", BRANCH,
         f"https://{token}@github.com/{REPO}.git", WORKDIR],
        capture_output=True, text=True,
    )
    # Scrub the token before anything reaches notebook output, which is saved
    # with the file.
    print((result.stdout + result.stderr).replace(token, "***"))
    if result.returncode != 0:
        raise RuntimeError("git clone failed -- check GITHUB_TOKEN has read access to the repo")
    # Drop the credential from .git/config too, so later git calls can't leak it.
    subprocess.run(
        ["git", "-C", WORKDIR, "remote", "set-url", "origin", f"https://github.com/{REPO}.git"],
        check=True,
    )

os.chdir(WORKDIR)
print("HEAD:", subprocess.run(["git", "log", "-1", "--oneline"],
                              capture_output=True, text=True).stdout.strip())

In [ ]:
# requirements-colab.txt first, then the package with --no-deps: installing
# soamp's own pinned dependency set would replace Colab's CUDA-matched torch
# build and drag in curation-only packages this workload never imports.
!pip install -q -r scripts/colab/requirements-colab.txt
!pip install -q -e . --no-deps

In [ ]:
import torch

from soamp.data.factory import build_dataset
from soamp.model.factory import build_model
from soamp.utils.device import resolve_device

device = resolve_device("auto")
print(f"torch {torch.__version__} | cuda available: {torch.cuda.is_available()} | device: {device}")
if device.type != "cuda":
    print("\nWARNING: running on CPU. The classifier is small enough not to care, but the "
          "PeptideCLM featurization pass will take ~15min instead of seconds.")

## 2. PeptideCLM features (the one GPU-bound step)

Three of the four feature artifacts are committed to the repo and arrive with
the clone — the RDKit descriptors (1.4 MB) and both organism representations
(the k-mer one has its genome vectors baked in, so no FASTAs are needed here).

The fourth, PeptideCLM's 12,371 × 768 embedding matrix, is ~190 MB — past
GitHub's per-file limit — so it is built per-environment. It takes seconds on a
GPU versus ~5 min on CPU. Drive caches it so a later session skips the work.

In [ ]:
import os
import shutil

from google.colab import drive

drive.mount("/content/drive")

CACHE_DIR = "/content/drive/MyDrive/soamp_cache"
RESULTS_DIR = "/content/drive/MyDrive/soamp_results"
os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

# Both are products of the same featurization pass, so they are cached together
# -- a scaler fitted on different embeddings than the ones on disk would be a
# silent correctness bug, not a crash.
PEPTIDECLM_ARTIFACTS = [
    "peptide_features_peptideclm.csv",
    "peptide_feature_scaler_peptideclm.json",
]

In [ ]:
cached = [f for f in PEPTIDECLM_ARTIFACTS if os.path.exists(f"{CACHE_DIR}/{f}")]

if len(cached) == len(PEPTIDECLM_ARTIFACTS):
    for name in PEPTIDECLM_ARTIFACTS:
        shutil.copy(f"{CACHE_DIR}/{name}", f"data/{name}")
        print(f"restored from Drive: {name} ({os.path.getsize(f'data/{name}') / 1e6:.1f} MB)")
else:
    if cached:
        print(f"partial cache ({cached}) -- rebuilding both so they stay consistent")
    print("building PeptideCLM embeddings...")
    !python pipeline/features/01_build_peptide_features.py --config config/features/peptide_peptideclm.yaml
    !python pipeline/features/03_fit_peptide_scaler.py --config config/features/peptide_peptideclm.yaml
    for name in PEPTIDECLM_ARTIFACTS:
        shutil.copy(f"data/{name}", f"{CACHE_DIR}/{name}")
        print(f"cached to Drive: {name}")

## 3. Are all four artifact sets present?

Each grid cell reads its own method-suffixed filenames, so all four coexist in
`data/` without overwriting one another. Checking here turns a missing artifact
into one clear message now rather than a `FileNotFoundError` three runs deep.

In [ ]:
import json

REQUIRED_ARTIFACTS = {
    "shared": [
        "mic_classification_dataset.csv",
        "val_split.json",
        "train_folds_leiden.csv",
    ],
    "peptide: rdkit_descriptors": [
        "peptide_features_rdkit.csv",
        "peptide_feature_scaler_rdkit.json",
    ],
    "peptide: peptideclm_embedding": PEPTIDECLM_ARTIFACTS,
    "organism: vocab_embedding": ["organism_vocab_vocab_embedding.json"],
    "organism: kmer_composition": ["organism_vocab_kmer_composition.json"],
}

missing = []
for group, names in REQUIRED_ARTIFACTS.items():
    for name in names:
        path = f"data/{name}"
        if os.path.exists(path):
            print(f"  {group:34s} {name:42s} {os.path.getsize(path) / 1e6:8.2f} MB")
        else:
            missing.append(f"{group}: {name}")

if missing:
    raise FileNotFoundError("missing artifacts:\n  " + "\n  ".join(missing))

# The two organism artifacts are self-describing -- confirm each really carries
# the method its filename claims, so a stale copy can't silently mislabel a run.
for name, expected in [("organism_vocab_vocab_embedding.json", "vocab_embedding"),
                       ("organism_vocab_kmer_composition.json", "kmer_composition")]:
    actual = json.load(open(f"data/{name}"))["method"]
    assert actual == expected, f"{name} says method={actual!r}, expected {expected!r}"

print("\nall artifacts present and self-consistent")

## 4. Run the grid (5-fold CV)

Each config is executed as a subprocess, exactly as it would run locally. The
runs are sequential and independent — if one fails, the others still complete
and the failure is reported at the end rather than killing the notebook.

`pipeline/train_cv.py` resolves `device: auto` itself, seeds from config, and
loops 5 folds over `data/train_folds_leiden.csv` — fitting on 4 folds and
evaluating on the held-out 5th each time. It never touches the `test` split
and saves no checkpoints; only `reports/cv_results_<exp_id>.csv` (row-level)
and `reports/train_cv_<exp_id>_log.txt` (summary) are written.

In [ ]:
import subprocess
import time

EXPERIMENTS = [
    "cv_rdkit_vocab",
    "cv_rdkit_kmer",
    "cv_peptideclm_vocab",
    "cv_peptideclm_kmer",
]

run_status = {}
for name in EXPERIMENTS:
    print(f"\n{'=' * 72}\n  {name}\n{'=' * 72}")
    started = time.time()
    result = subprocess.run(
        ["python", "pipeline/train_cv.py", "--config", f"config/train/{name}.yaml"],
        capture_output=True, text=True,
    )
    elapsed = time.time() - started
    # train_cv.py logs through stdlib logging -> stderr; tail it so a failure is
    # readable without scrolling the whole per-fold log.
    print("\n".join(result.stderr.strip().splitlines()[-25:]))
    run_status[name] = {"returncode": result.returncode, "seconds": round(elapsed, 1)}
    print(f"\n-> exit {result.returncode} in {elapsed:.0f}s")

failed = [n for n, s in run_status.items() if s["returncode"] != 0]
print(f"\n\n{len(EXPERIMENTS) - len(failed)}/{len(EXPERIMENTS)} runs succeeded")
if failed:
    print(f"FAILED: {failed}")

## 5. Compare the four runs

Pulled back from wandb rather than re-read off local files, so this is the same
record the runs actually published. Cross-fold `{fit,val}_<metric>_{mean,std}`
values live in each run's summary, written after `summarize_cv_metrics`
aggregates all 5 folds. There is no `test_*` key here — CV never touches test
— and no per-organism breakdown, since `train_and_evaluate_fold` only computes
overall `fit`/`val` metrics per fold.

In [ ]:
import pandas as pd
import wandb

PROJECT = "soamp"
GROUP = "featurization_grid_cv_v1"

api = wandb.Api()
entity = os.environ.get("WANDB_ENTITY") or api.default_entity
runs = api.runs(f"{entity}/{PROJECT}", filters={"group": GROUP})

records = []
for run in runs:
    summary, config = run.summary, run.config
    records.append({
        "run": run.name,
        "peptide": config.get("peptide_method"),
        "peptide_dim": config.get("peptide_feature_dim"),
        "organism": config.get("organism_method"),
        "organism_kind": config.get("organism_output_kind"),
        "device": config.get("device"),
        "val_auroc_mean": summary.get("val_auroc_mean"),
        "val_auroc_std": summary.get("val_auroc_std"),
        "val_accuracy_mean": summary.get("val_accuracy_mean"),
        "val_f1_mean": summary.get("val_f1_mean"),
        "val_precision_mean": summary.get("val_precision_mean"),
        "val_recall_mean": summary.get("val_recall_mean"),
        "fit_auroc_mean": summary.get("fit_auroc_mean"),
        "state": run.state,
    })

results_df = pd.DataFrame(records).sort_values("val_auroc_mean", ascending=False)
results_df

In [ ]:
import matplotlib.pyplot as plt

plot_df = results_df.dropna(subset=["val_auroc_mean"]).set_index("run").sort_values("val_auroc_mean")

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.barh(plot_df.index, plot_df["val_auroc_mean"], xerr=plot_df["val_auroc_std"], color="#4C78A8")
ax.set_xlim(0.5, 1.0)
ax.set_xlabel("val AUROC (mean \u00b1 std across 5 folds)")
ax.set_title("Cross-fold validation (train split only)")
ax.grid(axis="x", alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Save results off the runtime

No checkpoints exist for CV runs (`pipeline/train_cv.py` never invokes a
`Checkpointer`) — only the per-fold results CSV and log files, which live on
the Colab VM's disk and are discarded when the session ends. wandb already
holds the summary metrics; this keeps the row-level fold results.

In [ ]:
import glob

saved = []
for path in glob.glob("reports/cv_results_*.csv") + glob.glob("reports/train_cv_*_log.txt"):
    destination = f"{RESULTS_DIR}/{os.path.basename(path)}"
    shutil.copy(path, destination)
    saved.append(os.path.basename(path))

results_df.to_csv(f"{RESULTS_DIR}/featurization_grid_cv_results.csv", index=False)
print(f"copied {len(saved)} files + featurization_grid_cv_results.csv to {RESULTS_DIR}")

## Reading the result

This is a **cross-fold generalization estimate over the train split**, not
the held-out test benchmark — see `02_run_experiments.ipynb` for that. Two
things this grid does not settle, on top of the caveats that notebook
already lists:

- **5 epochs per fold, not 30.** This is a faster diagnostic
  (`config/train/cv_*.yaml` overrides `loop.epochs: 5`), not a fully-trained
  comparison. A cell that looks weaker here may still close the gap given the
  full 30 epochs `02` uses.
- **`val` here means cross-fold held-out folds of `train`, not the
  `data/val_split.json` validation slice `pipeline/train.py` uses.** The two
  are different validation mechanisms over overlapping but not identical row
  sets — don't compare a `val_auroc_mean` from this notebook directly against
  a `val_auroc` logged by `02`.